# FGSM attack on ResNet-18 with CIFAR-10

This notebook trains a CIFAR-adapted ResNet-18 or reloads `cifar10_resnet18.pt` when it already exists, then evaluates Fast Gradient Sign Method (FGSM) attacks.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

SEED = 42
BATCH_SIZE = 128
EPOCHS = 20
LEARNING_RATE = 0.1
EPSILONS = [0.0, 0.01, 0.03, 0.05]
DATA_DIR = Path("data")
CHECKPOINT = Path("cifar10_resnet18.pt")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Using device: {DEVICE}")

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
CIFAR10_CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck")

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
test_transform = transforms.ToTensor()
train_set = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=train_transform)
test_set = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_transform)
loader_options = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(train_set, shuffle=True, **loader_options)
test_loader = DataLoader(test_set, shuffle=False, **loader_options)

In [ ]:
class NormalizedModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(CIFAR10_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(CIFAR10_STD).view(1, 3, 1, 1))

    def forward(self, images):
        return self.model((images - self.mean) / self.std)


def make_model():
    resnet = models.resnet18(weights=None, num_classes=10)
    resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    resnet.maxpool = nn.Identity()
    return NormalizedModel(resnet)


model = make_model().to(DEVICE)

In [ ]:
def train_model(model, loader, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for epoch in range(1, epochs + 1):
        model.train()
        correct = total = 0
        loss_sum = 0.0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * labels.size(0)
            correct += logits.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        print(f"Epoch {epoch:3d}/{epochs}: loss={loss_sum / total:.4f}, accuracy={100 * correct / total:.2f}%")

In [ ]:
# Automatically reuse the checkpoint; train only if it does not exist.
if CHECKPOINT.exists():
    try:
        state_dict = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
    except TypeError:  # Compatibility with older PyTorch versions
        state_dict = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint: {CHECKPOINT}")
else:
    print(f"No checkpoint found at {CHECKPOINT}; starting training.")
    train_model(model, train_loader)
    torch.save(model.state_dict(), CHECKPOINT)
    print(f"Saved checkpoint: {CHECKPOINT}")

## Generate and evaluate FGSM adversarial examples

The remaining code computes the gradient of the classification loss with respect to each input image. FGSM uses the sign of this gradient to perturb the image by a chosen amount $\epsilon$, then clips the result to the valid pixel range $[0, 1]$.

The evaluation cell measures the model's accuracy on these adversarial images for several epsilon values and collects examples where a correctly classified clean image becomes misclassified. The final cell displays those successful attacks alongside their original images.

In [ ]:
def fgsm_attack(images, epsilon, gradients):
    return torch.clamp(images + epsilon * gradients.sign(), 0.0, 1.0).detach()


def evaluate_fgsm(model, loader, epsilon, max_examples=5):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    correct = total = 0
    examples = []
    for parameter in model.parameters():
        parameter.requires_grad_(False)

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        images.requires_grad_(True)
        clean_logits = model(images)
        gradients = torch.autograd.grad(criterion(clean_logits, labels), images)[0]
        adversarial = fgsm_attack(images, epsilon, gradients)
        with torch.no_grad():
            predictions = model(adversarial).argmax(1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)

        successful = clean_logits.detach().argmax(1).eq(labels) & predictions.ne(labels)
        for index in successful.nonzero(as_tuple=False).flatten():
            if len(examples) >= max_examples:
                break
            examples.append((images[index].detach().cpu(), adversarial[index].cpu(),
                             labels[index].item(), predictions[index].item()))

    for parameter in model.parameters():
        parameter.requires_grad_(True)
    return correct / total, examples

In [ ]:
results = {}
example_epsilon, examples = None, []
for epsilon in EPSILONS:
    accuracy, current_examples = evaluate_fgsm(model, test_loader, epsilon)
    results[epsilon] = accuracy
    print(f"epsilon={epsilon:.4f}  test_accuracy={100 * accuracy:.2f}%")
    if current_examples:
        example_epsilon, examples = epsilon, current_examples

In [ ]:
if examples:
    figure, axes = plt.subplots(2, len(examples), figsize=(2.5 * len(examples), 5), squeeze=False)
    for column, (clean, adversarial, true_label, predicted_label) in enumerate(examples):
        axes[0, column].imshow(clean.permute(1, 2, 0).numpy())
        axes[0, column].set_title(f"Clean: {CIFAR10_CLASSES[true_label]}")
        axes[1, column].imshow(adversarial.permute(1, 2, 0).numpy())
        axes[1, column].set_title(f"FGSM: {CIFAR10_CLASSES[predicted_label]}")
        axes[0, column].axis("off")
        axes[1, column].axis("off")
    figure.suptitle(f"Successful FGSM attacks (epsilon={example_epsilon:g})")
    figure.tight_layout()
    plt.show()
else:
    print("No successful attacks found to visualize.")